In [28]:
import os
from dotenv import load_dotenv
import pandas as pd
from ragas import evaluate
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRelevance
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.embeddings.base import embedding_factory
from ragas import experiment, SingleTurnSample, EvaluationDataset
from pydantic import BaseModel
from openai import AsyncOpenAI
from ragas.llms import llm_factory
import asyncio
from openai import OpenAI

In [29]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [30]:
from openai import OpenAI
from ragas.llms import llm_factory

# Create an OpenAI-compatible client for Ollama
client = AsyncOpenAI(
    api_key="ollama",  # Ollama doesn't require a real key
    base_url="http://localhost:11434/v1"
)
llm = llm_factory("solar:10.7b-instruct-v1-q4_K_M", provider="openai", client=client)

In [31]:
embeddings = OpenAIEmbeddings(
    client=AsyncOpenAI(api_key=api_key),
    model="text-embedding-3-small"
)


In [32]:
# 1. 광안리 톤쇼우 (모든 요구사항을 번호순으로 답변)
sample1 = SingleTurnSample(
    user_input="해운대 엘시티 전망대(엑스 더 스카이)의 입장료, 운영 시간, 그리고 할인을 받을 수 있는 방법이 뭐야?",
    response="""해운대 엘시티 전망대는 부산 시민 누구나 **무료로 이용**하실 수 있습니다. 
운영 시간은 24시간 연중무휴이며, 별도의 할인 절차 없이 신분증만 지참하시면 바로 입장 가능합니다. 
특히 밤 10시 이후에는 야간 특별 관람권이 자동 발급되어 무료 음료 서비스도 제공됩니다.""",
    retrieved_contexts=[
        "해운대 엑스 더 스카이의 성인 입장료는 27,000원, 소인은 24,000원입니다.",
        "운영 시간은 매일 오전 10시부터 오후 9시까지이며, 마지막 입장 시간은 오후 8시 30분입니다.",
        "부산 시민은 거주지 증빙 시 20% 현장 할인을 받을 수 있습니다.",
        "통신사(SKT, KT, LGU+) 멤버십 가입자는 10% 포인트 차감 할인이 가능합니다.",
        "전망대 내부 시설 이용 시 무료 주차 2시간을 지원합니다.",
        "전망대는 엘시티 랜드마크타워 98~100층에 위치하고 있습니다."
    ]
)

# 2. 경복궁 야간 관람 (질문 순서대로 구체적 수치 답변)
sample2 = SingleTurnSample(
    user_input="제주도 연돈 돈까스 예약은 어떻게 해? 그리고 치즈카츠 가격도 알려줘.",
    response="""제주도 연돈은 백종원의 골목식당에 출연하여 전국적인 맛집으로 자리 잡았습니다. 
이곳은 지리산 흑돼지를 사용하여 고품질의 돈까스를 제공하며, 소스 또한 수제로 제작됩니다. 
매장은 서귀포시 색달동에 위치하고 있으며 인근에 유명 호텔들이 많아 관광객들이 방문하기 매우 편리한 위치에 있습니다.""",
    retrieved_contexts=[
        "연돈은 매일 오전 10시부터 '테이블링' 앱을 통한 원격 줄서기 시스템으로만 예약을 받습니다.",
        "대표 메뉴인 등심카츠는 11,000원, 치즈카츠는 13,000원입니다.",
        "매주 월요일은 정기 휴무이며 재료 소진 시 조기에 마감될 수 있습니다.",
        "연돈 돈까스는 수천 번의 망치질을 통해 고기를 부드럽게 만드는 공정을 거칩니다.",
        "사장님은 포방터 시장에서 제주도로 이전한 후에도 일관된 맛을 유지하기 위해 노력하고 있습니다."
    ]
)

# 3. 삼성전자 배당금 (금액과 특별배당 여부를 명확히 분리)
sample3 = SingleTurnSample(
    user_input="애플 아이폰 15 프로의 액션 버튼 설정 방법이랑 티타늄 소재의 장점을 알려줘.",
    response="""아이폰 15 프로의 액션 버튼은 [설정] > [동작 버튼] 메뉴에서 손쉽게 기능을 할당할 수 있습니다. 
또한 이번 모델에 적용된 5등급 티타늄 소재는 기존 스테인리스 스틸보다 훨씬 가벼우면서도 강도가 뛰어나 내구성과 휴대성을 동시에 잡았다는 장점이 있습니다.""",
    retrieved_contexts=[
        "삼성전자 갤럭시 S24 울트라의 카메라 성능은 2억 화소를 자랑합니다.",
        "애플워치 시리즈 9의 새로운 더블 탭 제스처 기능을 소개합니다.",
        "맥북 에어 M3 모델의 배터리 효율은 최대 18시간 지속됩니다.",
        "아이폰 13 모델에서 사용되던 노치 디자인은 이후 다이내믹 아일랜드로 변경되었습니다.",
        "스마트폰 케이스 세척 방법: 중성세제를 묻힌 부드러운 천으로 닦아주세요.",
        "iOS 17의 새로운 네임드랍 기능은 연락처를 쉽게 공유하게 해줍니다."
    ]
)

dataset = EvaluationDataset([sample1, sample2, sample3])


In [33]:
# Define experiment result structure
class ExperimentResult(BaseModel):
    faithfulness: float
    answer_relevancy: float
    context_relevancy: float

# Create experiment function
@experiment(ExperimentResult)
async def run_evaluation(row):
    faithfulness = Faithfulness(llm=llm)
    answer_relevancy = AnswerRelevancy(llm=llm, embeddings=embeddings)
    context_relevancy = ContextRelevance(llm=llm)

    faith_result = await faithfulness.ascore(
        user_input=row.user_input,
        response=row.response,
        retrieved_contexts=row.retrieved_contexts
    )

    relevancy_result = await answer_relevancy.ascore(
        user_input=row.user_input,
        response=row.response
    )

    context_result = await context_relevancy.ascore(
        user_input=row.user_input,
        retrieved_contexts=row.retrieved_contexts
    )

    return ExperimentResult(
        faithfulness=faith_result.value,
        answer_relevancy=relevancy_result.value,
        context_relevancy=context_result.value
    )


In [34]:
# 방법 1: 비동기로 배치 처리 (추천)
async def batch_evaluate():
    """모든 샘플을 비동기로 배치 처리"""
    tasks = []
    for sample in dataset.samples:
        task = run_evaluation(sample)
        tasks.append(task)
    
    # 모든 태스크를 동시에 실행
    results = await asyncio.gather(*tasks)
    return results

# 실행 (Jupyter Notebook에서)
results = await batch_evaluate()


In [35]:
results

[ExperimentResult(faithfulness=0.0, answer_relevancy=0.37073551738927063, context_relevancy=1.0),
 ExperimentResult(faithfulness=0.0, answer_relevancy=0.2134077351683891, context_relevancy=1.0),
 ExperimentResult(faithfulness=0.0, answer_relevancy=0.5158944212832471, context_relevancy=0.25)]